# W02 — 파이썬 기초 ② : 자료구조

> **Ping 데이터:** 쇼핑몰 고객 데이터 (`Mall_Customers.csv`)
> **Pong 데이터:** COVID-19 국가별 현황 (`compact.csv`) — Our World in Data
> **학습 목표:** 여러 값을 하나의 변수에 체계적으로 저장하고 꺼낼 수 있다.

| 섹션 | 내용 | Ping-Pong |
|:---|:---|:---:|
| 1 | 리스트(List) | Ping 1 / Pong 1~3 |
| 2 | 리스트 메서드 | Ping 2 / Pong 4~5 |
| 3 | 튜플(Tuple) | Ping 3 / Pong 6~7 |
| 4 | 딕셔너리(Dictionary) | Ping 4 / Pong 8~10 |
| 5 | 집합(Set) | Ping 5 / Pong 11~12 |
| 6 | 불리언(Boolean) | Ping 6 / Pong 13~14 |
| 7 | 종합 실습 | Ping 7 / Pong 15~16 |
| 8 | 연습문제 | 17~26 |

---
## 실습 데이터 로드

아래 셀을 **가장 먼저 실행**하라.
`pandas`는 W04에서 자세히 배운다. 지금은 **그냥 실행만** 하면 된다.

**Mall_Customers.csv 컬럼:**

| 컬럼 | 설명 | 값 |
|:---|:---|:---|
| `CustomerID` | 고객 ID | 정수 |
| `Gender` | 성별 | Male / Female |
| `Age` | 나이 | 세 |
| `Annual Income (k$)` | 연간 소득 | 천달러 |
| `Spending Score (1-100)` | 소비 점수 | 1~100 |

In [ ]:
# ┌─────────────────────────────────────────────────────────┐
# │  아래 코드는 지금 몰라도 된다. 그냥 실행만 하라.         │
# │  pandas와 데이터 로드는 W04에서 자세히 배운다.           │
# └─────────────────────────────────────────────────────────┘
import pandas as pd

BASE = "https://raw.githubusercontent.com/leina99-lab/classes/main/AI%ED%94%84%EB%A1%9C%EA%B7%B8%EB%9E%98%EB%B0%8D/data/"

# ── Ping: Mall_Customers.csv 로드 → 처음 5명 추출 ────────────
df_mall   = pd.read_csv(BASE + "Mall_Customers.csv")
sample    = df_mall.iloc[:5]   # 처음 5명

# 리스트로 변환 (오늘 배울 자료구조!)
customer_ids    = list(sample["CustomerID"])
genders         = list(sample["Gender"])
ages            = list(sample["Age"])
incomes         = list(sample["Annual Income (k$)"])
spending_scores = list(sample["Spending Score (1-100)"])

print(f"[ Mall_Customers.csv ] 총 {len(df_mall)}명 로드 완료")
print(f"customer_ids    = {customer_ids}")
print(f"ages            = {ages}")
print(f"spending_scores = {spending_scores}")
print()

# ── Pong: compact.csv 로드 → 주요 국가 데이터 추출 ───────────
URL   = "https://catalog.ourworldindata.org/garden/covid/latest/compact/compact.csv"
df    = pd.read_csv(URL)

def safe_int(x):   return int(x)   if pd.notna(x) else 0
def safe_float(x): return float(x) if pd.notna(x) else 0.0

def get_latest(code):
    sub = df[(df["code"]==code) & (df["total_cases"].notna())]
    return sub.sort_values("date").iloc[-1] if len(sub) > 0 else None

kr = get_latest("KOR")
us = get_latest("USA")
jp = get_latest("JPN")
uk = get_latest("GBR")
de = get_latest("DEU")

# 국가별 확진자 리스트
countries       = ["South Korea", "United States", "Japan", "United Kingdom", "Germany"]
total_cases_list= [safe_int(kr["total_cases"]), safe_int(us["total_cases"]),
                   safe_int(jp["total_cases"]), safe_int(uk["total_cases"]),
                   safe_int(de["total_cases"])]
total_deaths_list=[safe_int(kr["total_deaths"]), safe_int(us["total_deaths"]),
                   safe_int(jp["total_deaths"]), safe_int(uk["total_deaths"]),
                   safe_int(de["total_deaths"])]
populations     = [safe_int(kr["population"]), safe_int(us["population"]),
                   safe_int(jp["population"]), safe_int(uk["population"]),
                   safe_int(de["population"])]
gdps            = [safe_float(kr["gdp_per_capita"]), safe_float(us["gdp_per_capita"]),
                   safe_float(jp["gdp_per_capita"]), safe_float(uk["gdp_per_capita"]),
                   safe_float(de["gdp_per_capita"])]
continents      = [str(kr["continent"]), str(us["continent"]),
                   str(jp["continent"]), str(uk["continent"]),
                   str(de["continent"])]

print(f"[ compact.csv ] COVID 5개국 데이터 로드 완료")
print(f"countries        = {countries}")
print(f"total_cases_list = {[f'{v:,}' for v in total_cases_list]}")

---
# Section 1. 리스트(List) — 순서 있는 수정 가능한 목록

## 개념 설명

**리스트**(list)는 여러 값을 순서대로 담는 자료구조이다.
대괄호(`[]`)로 만들고, 인덱스로 값을 꺼낸다.

```python
scores = [82, 45, 72, 91, 60]
scores[0]    # 82   ← 첫 번째
scores[-1]   # 60   ← 마지막
scores[1:3]  # [45, 72]  ← 슬라이싱
len(scores)  # 5
```

**기본 함수:**
```python
len(scores)   # 5     ← 길이
max(scores)   # 91    ← 최댓값
min(scores)   # 45    ← 최솟값
sum(scores)   # 350   ← 합계
sorted(scores)# [45, 60, 72, 82, 91] ← 정렬된 새 리스트 (원본 유지!)
```

### Ping 1 — 쇼핑몰 고객 나이 리스트 분석

In [ ]:
print("=== Ping 1: 리스트 기초 ===")
print(f"ages = {ages}")
print()

# 인덱싱
print(f"첫 번째 고객 나이: {ages[0]}")
print(f"마지막 고객 나이: {ages[-1]}")
print(f"2~4번째 나이: {ages[1:4]}")
print()

# 기본 통계
print(f"고객 수:    {len(ages)}명")
print(f"최고 나이:  {max(ages)}세")
print(f"최저 나이:  {min(ages)}세")
print(f"평균 나이:  {sum(ages)/len(ages):.1f}세")
print(f"정렬(오름): {sorted(ages)}")
print(f"정렬(내림): {sorted(ages, reverse=True)}")

### Pong 1 — `total_cases_list`로 인덱싱, 기본 통계를 출력하라.

<details>
<summary>▶ 정답 보기</summary>

```python
print("=== Pong 1: 리스트 기초 ===")
print(f"total_cases_list = {total_cases_list}")

print(f"첫 번째 국가 확진자: {total_cases_list[0]:,}")
print(f"마지막 국가 확진자: {total_cases_list[-1]:,}")
print(f"2~4번째: {total_cases_list[1:4]}")

print(f"국가 수:     {len(total_cases_list)}개국")
print(f"최대 확진자: {max(total_cases_list):,}명")
print(f"최소 확진자: {min(total_cases_list):,}명")
print(f"합계:        {sum(total_cases_list):,}명")
print(f"정렬(오름):  {sorted(total_cases_list)}")
```
</details>

In [ ]:
print("=== Pong 1: 리스트 기초 ===")
print(f"total_cases_list = {total_cases_list}")
print()

print(f"첫 번째 국가 확진자: {total_cases_list[___]:,}")
print(f"마지막 국가 확진자: {total_cases_list[___]:,}")
print(f"2~4번째: {total_cases_list[___]}")
print()
print(f"국가 수:     {len(total_cases_list)}개국")
print(f"최대 확진자: {max(___):,}명")
print(f"최소 확진자: {min(___):,}명")
print(f"합계:        {sum(___):,}명")
print(f"정렬(오름):  {sorted(___)}")

### Pong 2 — `countries` 리스트에서 인덱싱과 슬라이싱으로 각 부분을 추출하라.

<details>
<summary>▶ 정답 보기</summary>

```python
print("=== Pong 2: countries 인덱싱 ===")
print(f"countries = {countries}")
print(f"[0]   : {countries[0]}")
print(f"[-1]  : {countries[-1]}")
print(f"[1:3] : {countries[1:3]}")
print(f"[::2] : {countries[::2]}")
print(f"길이  : {len(countries)}")
```
</details>

In [ ]:
print("=== Pong 2: countries 인덱싱 ===")
print(f"countries = {countries}")
print()
print(f"[0]   : {countries[___]}")        # 'South Korea'
print(f"[-1]  : {countries[___]}")        # 'Germany'
print(f"[1:3] : {countries[___]}")        # 미국, 일본
print(f"[::2] : {countries[___]}")        # 0, 2, 4번째
print(f"길이  : {len(countries)}")

### Pong 3 — `gdps` 리스트로 통계를 계산하고, 평균 이상인 국가 인덱스를 한 줄 if로 찾아라.

<details>
<summary>▶ 정답 보기</summary>

```python
print("=== Pong 3: gdps 통계 ===")
print(f"gdps = {gdps}")

avg_gdp = sum(gdps) / len(gdps)
print(f"평균 GDP: {avg_gdp:,.0f}$")
print(f"최고 GDP: {max(gdps):,.0f}$ (인덱스: {gdps.index(max(gdps))})")
print(f"최저 GDP: {min(gdps):,.0f}$ (인덱스: {gdps.index(min(gdps))})")

print(f"{countries[0]}: {'평균 이상' if gdps[0] >= avg_gdp else '평균 미만'}")
print(f"{countries[1]}: {'평균 이상' if gdps[1] >= avg_gdp else '평균 미만'}")
print(f"{countries[2]}: {'평균 이상' if gdps[2] >= avg_gdp else '평균 미만'}")
print(f"{countries[3]}: {'평균 이상' if gdps[3] >= avg_gdp else '평균 미만'}")
print(f"{countries[4]}: {'평균 이상' if gdps[4] >= avg_gdp else '평균 미만'}")
```
</details>

In [ ]:
print("=== Pong 3: gdps 통계 ===")
print(f"gdps = {gdps}")
print()

avg_gdp = sum(gdps) / len(gdps)
print(f"평균 GDP: {avg_gdp:,.0f}$")
print(f"최고 GDP: {max(gdps):,.0f}$ (인덱스: {gdps.index(max(gdps))})")
print(f"최저 GDP: {min(gdps):,.0f}$ (인덱스: {gdps.index(min(gdps))})")
print()

# 한 줄 if: 각 국가 GDP가 평균 이상인지 판정
print(f"{countries[0]}: {'평균 이상' if gdps[___] >= avg_gdp else '평균 미만'}")
print(f"{countries[1]}: {'평균 이상' if gdps[___] >= avg_gdp else '평균 미만'}")
print(f"{countries[2]}: {'평균 이상' if gdps[___] >= avg_gdp else '평균 미만'}")
print(f"{countries[3]}: {'평균 이상' if gdps[___] >= avg_gdp else '평균 미만'}")
print(f"{countries[4]}: {'평균 이상' if gdps[___] >= avg_gdp else '평균 미만'}")

---
# Section 1-2. 리스트 슬라이싱(Slicing) 심화

## 개념 설명

슬라이싱은 리스트의 특정 구간을 잘라내는 기능이다.

```
리스트:  10  20  30  40  50  60  70  80
인덱스:   0   1   2   3   4   5   6   7
음수:    -8  -7  -6  -5  -4  -3  -2  -1
```

| 문법 | 하는 일 |
|:---|:---|
| `a[start:end]` | start부터 end-1까지 (끝 인덱스 미포함!) |
| `a[start:]` | start부터 끝까지 |
| `a[:end]` | 처음부터 end-1까지 |
| `a[:]` | 전체 복사 |
| `a[start:end:step]` | step만큼 건너뛰며 |
| `a[-2:]` | 뒤에서 2개 |
| `a[:-2]` | 끝의 2개 제외한 전체 |
| `a[::-1]` | 전체를 역순으로 |

> **끝 인덱스는 포함되지 않는다!** `a[1:5]`는 1, 2, 3, 4번 인덱스이다. 5번은 포함되지 않는다.

### Ping 1-2 — 리스트 슬라이싱 전체 패턴

In [ ]:
print("=== Ping 1-2: 리스트 슬라이싱 심화 ===")

a = [10, 20, 30, 40, 50, 60, 70, 80]
print(f"원본: {a}")
print()

# 기본 슬라이싱
print(f"a[1:5]    = {a[1:5]}")      # [20, 30, 40, 50]
print(f"a[0:3]    = {a[0:3]}")      # [10, 20, 30]
print(f"a[5:]     = {a[5:]}")       # [60, 70, 80]
print(f"a[:3]     = {a[:3]}")       # [10, 20, 30]
print(f"a[:]      = {a[:]}")        # 전체 복사
print()

# 음수 인덱스 슬라이싱
print(f"a[-7:-2]  = {a[-7:-2]}")    # [20, 30, 40, 50, 60]
print(f"a[-7:]    = {a[-7:]}")      # [20, 30, 40, 50, 60, 70, 80]
print(f"a[:-2]    = {a[:-2]}")      # [10, 20, 30, 40, 50, 60]
print(f"a[-2:]    = {a[-2:]}")      # [70, 80]
print()

# 스텝 슬라이싱
print(f"a[0:8:3]  = {a[0:8:3]}")   # [10, 40, 70]  3칸씩
print(f"a[::2]    = {a[::2]}")     # [10, 30, 50, 70]  짝수인덱스
print(f"a[1::2]   = {a[1::2]}")    # [20, 40, 60, 80]  홀수인덱스
print(f"a[::-1]   = {a[::-1]}")    # [80,70,...,10]  뒤집기
print()

# 슬라이싱의 덧셈
print(f"a[:-2]+a[-2:] = {a[:-2]+a[-2:]}")  # 전체와 같음
print(f"a[-2:]+a[:-2] = {a[-2:]+a[:-2]}")  # 순서가 달라짐!

### Pong 1-2a — `total_cases_list`와 `countries`로 슬라이싱 패턴을 연습하라.

<details>
<summary>▶ 정답 보기</summary>

```python
print("=== Pong 1-2a: 슬라이싱 심화 ===")
c = total_cases_list
print(f"원본: {c}")

print(f"c[1:4]    = {c[1:4]}")
print(f"c[2:]     = {c[2:]}")
print(f"c[:3]     = {c[:3]}")
print(f"c[-2:]    = {c[-2:]}")
print(f"c[:-1]    = {c[:-1]}")
print(f"c[::-1]   = {c[::-1]}")
print(f"c[::2]    = {c[::2]}")

n = countries
print(f"앞 3개:  {n[:3]}")
print(f"뒤 2개:  {n[-2:]}")
print(f"역순:    {n[::-1]}")
```
</details>

In [ ]:
print("=== Pong 1-2a: 슬라이싱 심화 ===")
c = total_cases_list
print(f"원본: {c}")
print()

print(f"c[1:4]    = {c[___]}")    # 인덱스 1~3
print(f"c[2:]     = {c[___]}")    # 2번부터 끝
print(f"c[:3]     = {c[___]}")    # 처음 3개
print(f"c[-2:]    = {c[___]}")    # 마지막 2개
print(f"c[:-1]    = {c[___]}")    # 마지막 제외 전체
print(f"c[::-1]   = {c[___]}")    # 뒤집기
print(f"c[::2]    = {c[___]}")    # 짝수 인덱스
print()

# countries 슬라이싱
n = countries
print(f"앞 3개:  {n[___]}")
print(f"뒤 2개:  {n[___]}")
print(f"역순:    {n[___]}")

### Pong 1-2b — 슬라이싱을 활용하여 확진자 상위 3개국과 하위 3개국을 각각 추출하라.

<details>
<summary>▶ 정답 보기</summary>

```python
print("=== Pong 1-2b: 슬라이싱 응용 ===")
pairs = sorted(zip(total_cases_list, countries), reverse=True)
cases_sorted     = [p[0] for p in pairs]
countries_sorted = [p[1] for p in pairs]

print(f"정렬된 확진자: {cases_sorted}")

top3_cases     = cases_sorted[:3]
top3_countries = countries_sorted[:3]
bot3_cases     = cases_sorted[-3:]
bot3_countries = countries_sorted[-3:]

print(f"확진자 TOP3:    {top3_countries}")
print(f"확진자 BOTTOM3: {bot3_countries}")

mid_countries = countries_sorted[1:4]
print(f"중간 3개국:    {mid_countries}")
```
</details>

In [ ]:
print("=== Pong 1-2b: 슬라이싱 응용 ===")

# (확진자, 국가) 정렬
pairs = sorted(zip(total_cases_list, countries), reverse=True)
cases_sorted    = [p[0] for p in pairs]
countries_sorted= [p[1] for p in pairs]

print(f"정렬된 확진자: {cases_sorted}")
print()

# 슬라이싱으로 상위/하위 추출
top3_cases    = cases_sorted[___]     # 상위 3개
top3_countries= countries_sorted[___]
bot3_cases    = cases_sorted[___]     # 하위 3개
bot3_countries= countries_sorted[___]

print(f"확진자 TOP3:    {top3_countries}")
print(f"확진자 BOTTOM3: {bot3_countries}")
print()

# 중간 국가 (슬라이싱)
mid_countries = countries_sorted[___]  # 1번~3번 (2번째~4번째)
print(f"중간 3개국:    {mid_countries}")

---
# Section 1-3. 리스트 연산 — `+`, `*`, 비교

## 개념 설명

### 이어붙이기(`+`)와 반복(`*`)
```python
a = [1, 2, 3]
b = [4, 5, 6]

a + b       # [1, 2, 3, 4, 5, 6]  ← 이어붙이기
a * 3       # [1, 2, 3, 1, 2, 3, 1, 2, 3]  ← 반복
```

> **함정!** 리스트끼리 곱셈은 불가능하다.
> `a * b` → `TypeError: can't multiply sequence by non-int of type 'list'`

### 리스트 비교
```python
[1,2,3] == [1,2,3]   # True  ← 순서까지 같아야
[1,2,3] == [3,2,1]   # False ← 순서 다르면 False
[1,2,3] <  [1,2,4]   # True  ← 앞에서부터 하나씩 비교
```

### Ping 1-3 — 리스트 연산

In [ ]:
print("=== Ping 1-3: 리스트 연산 ===")

list1 = [11, 22, 33, 44]
list2 = [55, 66]
a_list = ['a', 'b', 'c']

# 이어붙이기 (+)
print(f"list1 + list2        = {list1 + list2}")
print(f"list1 + list2 + a_list = {list1 + list2 + a_list}")
print()

# 반복 (*)
print(f"a_list * 3 = {a_list * 3}")
print(f"[0] * 5    = {[0] * 5}")    # 초기화 패턴
print()

# 비교 (==)
print(f"[1,2,3] == [1,2,3]  : {[1,2,3] == [1,2,3]}")  # True
print(f"[1,2,3] == [3,2,1]  : {[1,2,3] == [3,2,1]}")  # False  순서 중요!
print()

# 크기 비교 (앞에서부터 하나씩)
list3 = [1, 2, 3, 4]
list4 = [4, 2, 1, 3]
print(f"[1,2,3,4] < [4,2,1,3]: {list3 < list4}")    # True (1<4)
print(f"[4,2,1,3] < [1,2,3,4]: {list4 < list3}")    # False

# 리스트끼리 곱셈은 오류!
try:
    result = list1 * list2
except TypeError as e:
    print(f"list1 * list2 오류: {e}")

### Pong 1-3a — `total_cases_list`와 `countries`를 이어붙이고, 비교 연산을 수행하라.

<details>
<summary>▶ 정답 보기</summary>

```python
print("=== Pong 1-3a: 리스트 연산 ===")
combined = total_cases_list + total_deaths_list
print(f"확진+사망 합친 리스트 길이: {len(combined)}")
print(f"합친 리스트: {combined}")

zeros = [0] * len(countries)
print(f"0으로 초기화: {zeros}")

copy = total_cases_list[:]
print(f"원본 == 복사본: {total_cases_list == copy}")

reversed_list = total_cases_list[::-1]
print(f"원본 == 역순:   {total_cases_list == reversed_list}")

print(f"total_cases_list > total_deaths_list: {total_cases_list > total_deaths_list}")
```
</details>

In [ ]:
print("=== Pong 1-3a: 리스트 연산 ===")

# 이어붙이기
combined = total_cases_list ___ total_deaths_list
print(f"확진+사망 합친 리스트 길이: {len(combined)}")
print(f"합친 리스트: {combined}")
print()

# 반복으로 초기화
zeros = [___] * len(countries)  # 0으로 초기화
print(f"0으로 초기화: {zeros}")
print()

# 비교
copy = total_cases_list[:]
print(f"원본 == 복사본: {total_cases_list ___ copy}")      # True

reversed_list = total_cases_list[::-1]
print(f"원본 == 역순:   {total_cases_list ___ reversed_list}")  # False (다른 순서)
print()

# 크기 비교 (첫 번째 값부터 비교)
print(f"total_cases_list > total_deaths_list: {total_cases_list ___ total_deaths_list}")

### Pong 1-3b — `countries`를 3번 반복한 리스트를 만들고, 원본과 반복본을 비교하라.

<details>
<summary>▶ 정답 보기</summary>

```python
print("=== Pong 1-3b: 반복과 비교 ===")
tripled = countries * 3
print(f"3번 반복: {tripled}")
print(f"길이: {len(tripled)}")

first_part = tripled[:5]
print(f"앞 5개: {first_part}")
print(f"원본과 같은가: {first_part == countries}")

sorted_countries = sorted(countries)
print(f"원본:   {countries}")
print(f"정렬후: {sorted_countries}")
print(f"같은가: {countries == sorted_countries}")
```
</details>

In [ ]:
print("=== Pong 1-3b: 반복과 비교 ===")

# 반복
tripled = countries ___ 3
print(f"3번 반복: {tripled}")
print(f"길이: {len(tripled)}")
print()

# 원본 리스트와 반복본의 앞부분 비교
first_part = tripled[___]  # 앞 5개 (원본과 같아야)
print(f"앞 5개: {first_part}")
print(f"원본과 같은가: {first_part ___ countries}")
print()

# 두 리스트 비교: 알파벳순으로 정렬된 국가명 비교
sorted_countries = sorted(countries)
print(f"원본:   {countries}")
print(f"정렬후: {sorted_countries}")
print(f"같은가: {countries ___ sorted_countries}")

---
# Section 1-4. 이중 리스트(Nested List)

## 개념 설명

리스트 안에 리스트를 넣을 수 있다.

```python
matrix = [[1, 2, 3],
           [4, 5, 6],
           [7, 8, 9]]

matrix[0]       # [1, 2, 3]    ← 첫 번째 행
matrix[1][2]    # 6            ← 2행 3열
len(matrix)     # 3            ← 행 수 (리스트 개수)
```

실제 데이터 분석에서는 **행(row) = 한 개의 데이터**, **열(column) = 특성값** 구조로 자주 등장한다.

### Ping 1-4 — 이중 리스트

In [ ]:
print("=== Ping 1-4: 이중 리스트 ===")

# 쇼핑몰 고객 데이터 이중 리스트
# [CustomerID, Age, Annual_Income, Spending_Score]
customers_matrix = [
    [customer_ids[0], ages[0], incomes[0], spending_scores[0]],
    [customer_ids[1], ages[1], incomes[1], spending_scores[1]],
    [customer_ids[2], ages[2], incomes[2], spending_scores[2]],
    [customer_ids[3], ages[3], incomes[3], spending_scores[3]],
    [customer_ids[4], ages[4], incomes[4], spending_scores[4]],
]

print(f"전체 행렬: {customers_matrix}")
print(f"행 수: {len(customers_matrix)}")
print()

# 특정 행 접근
print(f"첫 번째 고객 행: {customers_matrix[0]}")
print(f"마지막 고객 행: {customers_matrix[-1]}")
print()

# 특정 셀 접근 [행][열]
print(f"첫 번째 고객 ID:  {customers_matrix[0][0]}")
print(f"두 번째 고객 나이: {customers_matrix[1][1]}")
print(f"세 번째 고객 점수: {customers_matrix[2][3]}")
print()

# for로 이중 리스트 순회
print(f"{'ID':>5} {'나이':>5} {'소득':>8} {'점수':>6}")
print("-" * 28)
for row in customers_matrix:
    print(f"{row[0]:>5} {row[1]:>4}세 {row[2]:>6}k$ {row[3]:>5}점")

### Pong 1-4 — COVID 5개국 데이터를 이중 리스트로 만들고 특정 값에 접근하라.

<details>
<summary>▶ 정답 보기</summary>

```python
print("=== Pong 1-4: 이중 리스트 ===")
covid_matrix = [
    [countries[i], total_cases_list[i], total_deaths_list[i], populations[i]]
    for i in range(5)
]

print(f"행 수: {len(covid_matrix)}")
print(f"열 수: {len(covid_matrix[0])}")

print(f"첫 번째 국가명:      {covid_matrix[0][0]}")
print(f"미국 확진자:         {covid_matrix[1][1]:,}")
print(f"일본 사망자:         {covid_matrix[2][2]:,}")

print(f"{'국가':<20} {'확진자':>12} {'사망자':>10}")
print("-" * 44)
for row in covid_matrix:
    print(f"{row[0]:<20} {row[1]:>12,} {row[2]:>10,}")
```
</details>

In [ ]:
print("=== Pong 1-4: 이중 리스트 ===")

# [국가명, 확진자, 사망자, 인구] 이중 리스트 만들기
covid_matrix = [
    [countries[0], total_cases_list[0], total_deaths_list[0], populations[0]],
    [countries[1], total_cases_list[___], total_deaths_list[___], populations[___]],
    [countries[2], total_cases_list[___], total_deaths_list[___], populations[___]],
    [countries[3], total_cases_list[___], total_deaths_list[___], populations[___]],
    [countries[4], total_cases_list[___], total_deaths_list[___], populations[___]],
]

print(f"행 수: {len(covid_matrix)}")
print(f"열 수: {len(covid_matrix[0])}")
print()

# 특정 셀 접근
print(f"첫 번째 국가명:      {covid_matrix[___][___]}")
print(f"미국 확진자:         {covid_matrix[___][___]:,}")
print(f"일본 사망자:         {covid_matrix[___][___]:,}")
print()

# 이중 for로 순회
print(f"{'국가':<20} {'확진자':>12} {'사망자':>10}")
print("-" * 44)
for row in covid_matrix:
    print(f"{row[0]:<20} {row[1]:>12,} {row[2]:>10,}")

---
# Section 1-5. 슬라이싱·연산 추가 연습문제

### 추가 연습 1 — `[10,20,30,40,50,60,70,80]`에서 `a[0:8:3]`의 결과를 예측하고 확인하라.

<details>
<summary>▶ 정답 보기</summary>

```python
a = [10,20,30,40,50,60,70,80]
print(a[0:8:3])   # [10, 40, 70]
```
</details>

In [ ]:
# 여기에 직접 풀어보세요

### 추가 연습 2 — `total_cases_list`를 역순으로 뒤집은 새 리스트를 만들어라. (원본 유지)

<details>
<summary>▶ 정답 보기</summary>

```python
rev = total_cases_list[::-1]
print(rev)
print(total_cases_list)  # 원본 유지
```
</details>

In [ ]:
# 여기에 직접 풀어보세요

### 추가 연습 3 — `countries + ['Brazil','India']`로 새 7개국 리스트를 만들어라.

<details>
<summary>▶ 정답 보기</summary>

```python
all7 = countries + ['Brazil','India']
print(all7)
print(len(all7))
```
</details>

In [ ]:
# 여기에 직접 풀어보세요

### 추가 연습 4 — `[0]*5`로 초기화 리스트를 만들고 인덱스 2의 값만 99로 바꿔라.

<details>
<summary>▶ 정답 보기</summary>

```python
lst = [0]*5
lst[2] = 99
print(lst)   # [0, 0, 99, 0, 0]
```
</details>

In [ ]:
# 여기에 직접 풀어보세요

### 추가 연습 5 — 이중 리스트 `covid_matrix`에서 사망률이 가장 높은 국가를 찾아라.

<details>
<summary>▶ 정답 보기</summary>

```python
covid_matrix = [[countries[i],total_cases_list[i],total_deaths_list[i]] for i in range(5)]
drs = [row[2]/row[1]*100 for row in covid_matrix]
max_idx = drs.index(max(drs))
print(covid_matrix[max_idx][0], f'{max(drs):.3f}%')
```
</details>

In [ ]:
# 여기에 직접 풀어보세요

---
# Section 2. 리스트 메서드(Method)

## 개념 설명

| 메서드 | 기능 | 반환 |
|:---|:---|:---|
| `.append(x)` | 맨 끝에 추가 | `None` (원본 변경) |
| `.insert(i, x)` | i번 위치에 삽입 | `None` |
| `.remove(x)` | 값으로 삭제 | `None` |
| `.pop()` | 마지막 꺼내서 삭제 | 꺼낸 값 |
| `.sort()` | 원본 정렬 | `None` |
| `.reverse()` | 원본 뒤집기 | `None` |
| `.index(x)` | 값의 위치 | 인덱스 |
| `.count(x)` | 값의 개수 | 개수 |
| `x in list` | 포함 여부 | `True/False` |

> **함정!** `.append()`와 `.sort()`는 **원본을 바꾸고 `None`을 반환**한다.
> ```python
> result = scores.append(99)  # result = None !
> result = scores.sort()      # result = None !
> ```
> 새 리스트를 원하면 `sorted(scores)`를 사용한다.

### Ping 2 — 쇼핑몰 고객 소비점수 리스트 조작

In [ ]:
print("=== Ping 2: 리스트 메서드 ===")
scores = spending_scores.copy()   # 원본 보존을 위해 복사
print(f"원본: {scores}")
print()

# 추가
scores.append(75)
print(f"append(75): {scores}")

# 삽입
scores.insert(0, 50)
print(f"insert(0,50): {scores}")

# 삭제
scores.remove(50)
print(f"remove(50): {scores}")

# pop
last = scores.pop()
print(f"pop(): {last} 꺼냄 → {scores}")

# 정렬 (원본 변경)
scores.sort()
print(f"sort(): {scores}")

# 포함 여부
print(f"91 in scores: {91 in scores}")
print(f"100 in scores: {100 in scores}")

# index, count
print(f"index(72): {scores.index(72)}번 인덱스")

# 함정 시연!
print()
print("=== 함정: append 반환값 ===")
result = scores.append(88)
print(f"scores.append(88) 반환값: {result}")  # None!
print(f"scores: {scores}")  # 원본은 바뀜

### Pong 4 — `total_cases_list`를 복사한 후 추가, 삭제, 정렬을 수행하라.

<details>
<summary>▶ 정답 보기</summary>

```python
print("=== Pong 4: 리스트 메서드 ===")
cases = total_cases_list.copy()
print(f"원본 복사본: {cases}")

cases.append(36000000)
print(f"append 후: {cases}")

biggest = cases.pop()
print(f"pop(): {biggest:,} 꺼냄")
print(f"pop 후: {cases}")

cases.sort()
print(f"sort(): {cases}")

print(f"한국 확진자 수 포함 여부: {total_cases_list[0] in cases}")
```
</details>

In [ ]:
print("=== Pong 4: 리스트 메서드 ===")
cases = total_cases_list.___()   # 복사 (원본 보존!)
print(f"원본 복사본: {cases}")
print()

# 새 국가 확진자 수 추가 (브라질: 3,600만 가정)
cases.___(36000000)
print(f"append 후: {cases}")

# 가장 큰 값 꺼내서 삭제
biggest = cases.___()
print(f"pop(): {biggest:,} 꺼냄")
print(f"pop 후: {cases}")

# 오름차순 정렬
cases.___()
print(f"sort(): {cases}")

# 포함 여부
print(f"한국 확진자 수 포함 여부: {total_cases_list[0] ___ cases}")

### Pong 5 — 리스트 복사 함정을 직접 확인하라.

<details>
<summary>▶ 정답 보기</summary>

```python
print("=== Pong 5: 복사 함정 ===")
a = countries
a.append("Brazil")
print(f"a = {a}")
print(f"countries = {countries}")
print(f"a is countries: {a is countries}")

b = countries.copy()
b.append("India")
print(f"b = {b}")
print(f"countries = {countries}")
print(f"b is countries: {b is countries}")
```
</details>

In [ ]:
print("=== Pong 5: 복사 함정 ===")

# 잘못된 복사 (같은 리스트를 가리킴)
a = countries
a.___("Brazil")
print(f"a = {a}")
print(f"countries = {countries}")   # countries도 바뀐다!
print(f"a is countries: {a is countries}")
print()

# 올바른 복사
b = countries.___()    # 진짜 복사
b.append("India")
print(f"b = {b}")
print(f"countries = {countries}")  # countries는 그대로!
print(f"b is countries: {b is countries}")

---
# Section 3. 튜플(Tuple) — 수정 불가 리스트

## 개념 설명

**튜플**(tuple)은 한 번 만들면 **수정할 수 없는** 자료구조이다.
소괄호(`()`)로 만든다.

```python
# 리스트 vs 튜플
scores_list  = [82, 45, 72]   # 수정 가능
scores_tuple = (82, 45, 72)   # 수정 불가

scores_tuple[0] = 99   # TypeError! 수정 불가
```

**튜플 언패킹(unpacking):**
```python
customer = ("C001", "Male", 25, 82)
cid, gender, age, score = customer   # 한 번에 할당!
print(cid)    # "C001"
print(age)    # 25
```

> **언제 쓰는가?** 변경하면 안 되는 데이터(좌표, 고객 기본 정보 등)에 사용한다.

### Ping 3 — 고객 기본 정보를 튜플로 관리

In [ ]:
print("=== Ping 3: 튜플 ===")

# 고객 기본 정보 튜플 (변경 불가)
customer = (customer_ids[0], genders[0], ages[0], incomes[0], spending_scores[0])
print(f"고객 튜플: {customer}")
print(f"type: {type(customer)}")
print()

# 인덱싱 (읽기는 가능)
print(f"ID: {customer[0]}")
print(f"성별: {customer[1]}")
print(f"나이: {customer[2]}세")
print()

# 언패킹
cid, gender, age_val, income, score = customer
print(f"언패킹 결과:")
print(f"  cid={cid}, gender={gender}, age={age_val}, income={income}, score={score}")
print()

# 수정 시도 → 오류
try:
    customer[0] = 999
except TypeError as e:
    print(f"수정 시도 오류: {e}")

### Pong 6 — 5개국 데이터를 튜플로 만들고 언패킹하라.

<details>
<summary>▶ 정답 보기</summary>

```python
print("=== Pong 6: 튜플 ===")
kr_tuple = (countries[0], total_cases_list[0], total_deaths_list[0], populations[0], gdps[0])
print(f"한국 튜플: {kr_tuple}")
print(f"type: {type(kr_tuple)}")

print(f"국가명: {kr_tuple[0]}")
print(f"확진자: {kr_tuple[1]:,}")
print(f"사망자: {kr_tuple[2]:,}")

name, cases, deaths, pop, gdp = kr_tuple
print(f"언패킹: {name}, 확진 {cases:,}, 사망 {deaths:,}")
```
</details>

In [ ]:
print("=== Pong 6: 튜플 ===")

# 한국 데이터를 튜플로 만들기
kr_tuple = (countries[0], total_cases_list[0], total_deaths_list[0], populations[0], gdps[0])
print(f"한국 튜플: {kr_tuple}")
print(f"type: {type(kr_tuple)}")
print()

# 인덱싱
print(f"국가명: {kr_tuple[___]}")
print(f"확진자: {kr_tuple[___]:,}")
print(f"사망자: {kr_tuple[___]:,}")
print()

# 언패킹
name, cases, deaths, pop, gdp = ___
print(f"언패킹: {name}, 확진 {cases:,}, 사망 {deaths:,}")

### Pong 7 — 두 변수의 값을 튜플 언패킹으로 교환하라.

<details>
<summary>▶ 정답 보기</summary>

```python
print("=== Pong 7: 튜플 값 교환 ===")
a = total_cases_list[0]
b = total_cases_list[1]
print(f"교환 전: a={a:,}, b={b:,}")

a, b = b, a
print(f"교환 후: a={a:,}, b={b:,}")

date_str = "2023-01-15"
year, month, day = date_str.split("-")
print(f"연: {year}, 월: {month}, 일: {day}")
```
</details>

In [ ]:
print("=== Pong 7: 튜플 값 교환 ===")

# 두 국가 확진자 수 교환
a = total_cases_list[0]   # 한국
b = total_cases_list[1]   # 미국
print(f"교환 전: a={a:,}, b={b:,}")

# 튜플 언패킹으로 교환 (한 줄!)
___, ___ = b, a
print(f"교환 후: a={a:,}, b={b:,}")

# 날짜 분리도 튜플 언패킹으로
date_str = "2023-01-15"
year, month, day = date_str.___("_")   # split 활용
print(f"연: {year}, 월: {month}, 일: {day}")

---
# Section 4. 딕셔너리(Dictionary) — 키-값 쌍의 모음

## 개념 설명

**딕셔너리**(dict)는 `키(key): 값(value)` 쌍으로 데이터를 저장한다.
중괄호(`{}`)로 만들고, 키로 값을 꺼낸다.

```python
customer = {
    "id": 1,
    "name": "Alice",
    "age": 25,
    "score": 82
}

customer["name"]          # "Alice"
customer.get("email")     # None (없는 키 → 오류 없이 None)
customer.get("email", "없음")  # "없음" (기본값 지정)
```

> **함정!** 없는 키를 `[]`로 접근하면 `KeyError`!
> 안전하게 접근하려면 `.get(키)` 또는 `.get(키, 기본값)`을 사용한다.

### Ping 4 — 고객 딕셔너리 만들기와 조작

In [ ]:
print("=== Ping 4: 딕셔너리 ===")

# 고객 딕셔너리 생성
customer = {
    "id":      customer_ids[0],
    "gender":  genders[0],
    "age":     ages[0],
    "income":  incomes[0],
    "score":   spending_scores[0]
}
print(f"고객 정보: {customer}")
print()

# 값 접근
print(f"ID:    {customer['id']}")
print(f"나이:  {customer['age']}세")
print(f"점수:  {customer['score']}")
print()

# .get() 안전한 접근
print(f".get('gender')       : {customer.get('gender')}")
print(f".get('email')        : {customer.get('email')}")          # None
print(f".get('email','없음') : {customer.get('email', '없음')}")  # 기본값
print()

# 추가·수정·삭제
customer["grade"] = "VIP" if customer["score"] >= 80 else "일반"
print(f"grade 추가: {customer}")

customer["age"] = customer["age"] + 1
print(f"age 수정: {customer['age']}세")

del customer["grade"]
print(f"grade 삭제: {customer}")
print()

# 키/값 순회
print("키 목록:", list(customer.keys()))
print("값 목록:", list(customer.values()))

### Pong 8 — 한국 COVID 데이터를 딕셔너리로 만들고 조작하라.

<details>
<summary>▶ 정답 보기</summary>

```python
print("=== Pong 8: 딕셔너리 생성 ===")
kr_covid = {
    "country":     countries[0],
    "total_cases": total_cases_list[0],
    "total_deaths":total_deaths_list[0],
    "population":  populations[0],
    "gdp":         gdps[0]
}
print(f"한국 COVID: {kr_covid}")

print(f"국가: {kr_covid['country']}")
print(f"확진: {kr_covid['total_cases']:,}명")

death_rate = kr_covid["total_deaths"] / kr_covid["total_cases"] * 100
kr_covid["death_rate"] = round(death_rate, 3)
print(f"death_rate 추가 후: {kr_covid}")
```
</details>

In [ ]:
print("=== Pong 8: 딕셔너리 생성 ===")

# 한국 COVID 딕셔너리 생성
kr_covid = {
    "country":    countries[___],
    "total_cases":total_cases_list[___],
    "total_deaths":total_deaths_list[___],
    "population": populations[___],
    "gdp":        gdps[___]
}
print(f"한국 COVID: {kr_covid}")
print()

# 값 접근
print(f"국가: {kr_covid[___]}")
print(f"확진: {kr_covid[___]:,}명")
print()

# 사망률 계산 후 딕셔너리에 추가
death_rate = kr_covid["total_deaths"] / kr_covid["total_cases"] * 100
kr_covid[___] = round(death_rate, 3)
print(f"death_rate 추가 후: {kr_covid}")

### Pong 9 — 5개국 딕셔너리를 만들고 `.keys()`, `.values()`, `.items()`를 확인하라.

<details>
<summary>▶ 정답 보기</summary>

```python
print("=== Pong 9: 딕셔너리 키/값 ===")
cases_dict = {
    countries[0]: total_cases_list[0],
    countries[1]: total_cases_list[1],
    countries[2]: total_cases_list[2],
    countries[3]: total_cases_list[3],
    countries[4]: total_cases_list[4]
}
print(f"cases_dict = {cases_dict}")

print(f"키 목록:   {list(cases_dict.keys())}")
print(f"값 목록:   {list(cases_dict.values())}")

print(f"'South Korea' in cases_dict: {'South Korea' in cases_dict}")
print(f"'Brazil' in cases_dict:      {'Brazil' in cases_dict}")

print(f"한국 확진자: {cases_dict.get('South Korea', 0):,}")
print(f"브라질 확진자: {cases_dict.get('Brazil', '데이터 없음')}")
```
</details>

In [ ]:
print("=== Pong 9: 딕셔너리 키/값 ===")

# 국가명: 확진자수 딕셔너리
cases_dict = {
    countries[0]: total_cases_list[0],
    countries[1]: total_cases_list[1],
    countries[2]: total_cases_list[2],
    countries[3]: total_cases_list[3],
    countries[4]: total_cases_list[4]
}
print(f"cases_dict = {cases_dict}")
print()

print(f"키 목록:   {list(cases_dict.___)}")
print(f"값 목록:   {list(cases_dict.___)}")
print()

# 포함 여부
print(f"'South Korea' in cases_dict: {'South Korea' ___ cases_dict}")
print(f"'Brazil' in cases_dict:      {'Brazil' ___ cases_dict}")
print()

# .get() 활용
print(f"한국 확진자: {cases_dict.get('South Korea', 0):,}")
print(f"브라질 확진자: {cases_dict.get('Brazil', '데이터 없음')}")

### Pong 10 — 딕셔너리의 값을 수정·삭제하고 `KeyError`를 안전하게 처리하라.

<details>
<summary>▶ 정답 보기</summary>

```python
print("=== Pong 10: 딕셔너리 수정·삭제 ===")
info = {
    "country": "South Korea",
    "cases":   total_cases_list[0],
    "deaths":  total_deaths_list[0],
    "temp":    "삭제 예정"
}
print(f"초기: {info}")

info["cases"] = info["cases"] + 100
print(f"수정 후: {info['cases']:,}")

del info["temp"]
print(f"삭제 후: {info}")

removed = info.pop("deaths", "없음")
print(f"pop('deaths'): {removed:,}")
print(f"pop 후: {info}")

val = info.get("없는키", "기본값")
print(f"없는 키 .get(): {val}")
```
</details>

In [ ]:
print("=== Pong 10: 딕셔너리 수정·삭제 ===")

info = {
    "country": "South Korea",
    "cases":   total_cases_list[0],
    "deaths":  total_deaths_list[0],
    "temp":    "삭제 예정"
}
print(f"초기: {info}")

# 값 수정 (확진자 100명 추가)
info[___] = info["cases"] + 100
print(f"수정 후: {info['cases']:,}")

# 키 삭제
del info[___]
print(f"삭제 후: {info}")

# pop()으로 꺼내서 삭제
removed = info.pop(___, "없음")
print(f"pop('deaths'): {removed:,}")
print(f"pop 후: {info}")

# 없는 키 안전하게 처리
val = info.get("없는키", "기본값")
print(f"없는 키 .get(): {val}")

---
# Section 5. 집합(Set) — 중복 없는 값의 모음

## 개념 설명

**집합**(set)은 **중복을 허용하지 않는** 자료구조이다.
수학의 집합과 같은 개념이다.

```python
# 중복 자동 제거
visitors = {101, 102, 103, 102, 101}
print(visitors)  # {101, 102, 103}

# 리스트에서 중복 제거
ages = [25, 30, 25, 22, 30]
unique_ages = set(ages)  # {25, 30, 22}
```

**집합 연산:**
```python
a = {1, 2, 3, 4}
b = {3, 4, 5, 6}

a | b   # {1,2,3,4,5,6}  합집합
a & b   # {3,4}           교집합
a - b   # {1,2}           차집합
```

> **주의!** 집합은 **순서가 없다**. `set[0]` 인덱싱 불가!

### Ping 5 — 고객 성별 중복 제거와 집합 연산

In [ ]:
print("=== Ping 5: 집합 ===")

# 리스트에서 중복 제거
genders_set = set(genders)
print(f"genders      = {genders}")
print(f"set(genders) = {genders_set}  ← 중복 제거!")
print()

# 나이 중복 제거
ages_set = set(ages)
print(f"ages      = {ages}")
print(f"set(ages) = {ages_set}")
print()

# 집합 연산 예시
high_score  = {1, 2, 5}           # 소비점수 80 이상 고객 ID
young       = {1, 3, 4}           # 나이 30 미만 고객 ID

print(f"고소비:      {high_score}")
print(f"젊은층:      {young}")
print(f"합집합(|):   {high_score | young}")   # 둘 중 하나라도 해당
print(f"교집합(&):   {high_score & young}")   # 둘 다 해당
print(f"차집합(-):   {high_score - young}")   # 고소비이지만 젊지 않음
print()

# 포함 여부
print(f"1 in high_score: {1 in high_score}")
print(f"6 in high_score: {6 in high_score}")

### Pong 11 — `continents` 리스트에서 중복을 제거하고 집합 연산을 수행하라.

<details>
<summary>▶ 정답 보기</summary>

```python
print("=== Pong 11: 집합 ===")
cont_set = set(continents)
print(f"continents = {continents}")
print(f"set 변환   = {cont_set}")
print(f"고유 대륙 수: {len(cont_set)}개")

asia_idx   = {i for i in range(len(continents)) if continents[i] == "Asia"}
europe_idx = {i for i in range(len(continents)) if continents[i] == "Europe"}

print(f"아시아 인덱스: {asia_idx}")
print(f"유럽 인덱스:   {europe_idx}")
print(f"합집합:        {asia_idx | europe_idx}")
print(f"교집합:        {asia_idx & europe_idx}")

print(f"'Asia' in cont_set: {'Asia' in cont_set}")
print(f"'Africa' in cont_set: {'Africa' in cont_set}")
```
</details>

In [ ]:
print("=== Pong 11: 집합 ===")

# 대륙 중복 제거
cont_set = ___(continents)
print(f"continents = {continents}")
print(f"set 변환   = {cont_set}  ← 중복 제거!")
print(f"고유 대륙 수: {len(cont_set)}개")
print()

# 아시아 국가와 유럽 국가 집합 (인덱스로 구성)
asia_idx    = {i for i in range(len(continents)) if continents[i] == "Asia"}
europe_idx  = {i for i in range(len(continents)) if continents[i] == "Europe"}

print(f"아시아 인덱스: {asia_idx}")
print(f"유럽 인덱스:   {europe_idx}")
print(f"합집합:        {asia_idx ___ europe_idx}")
print(f"교집합:        {asia_idx ___ europe_idx}")
print()

# 포함 여부
print(f"'Asia' in cont_set: {'Asia' ___ cont_set}")
print(f"'Africa' in cont_set: {'Africa' ___ cont_set}")

### Pong 12 — 소비점수 리스트에서 중복 제거, 80점 이상과 50점 미만 집합을 만들고 연산하라.

<details>
<summary>▶ 정답 보기</summary>

```python
print("=== Pong 12: 집합 연산 응용 ===")
score_set = set(spending_scores)
print(f"spending_scores = {spending_scores}")
print(f"중복 제거       = {score_set}")

high = {i for i in range(len(spending_scores)) if spending_scores[i] >= 80}
low  = {i for i in range(len(spending_scores)) if spending_scores[i] < 50}

print(f"80점 이상 인덱스: {high}")
print(f"50점 미만 인덱스: {low}")
print(f"합집합 (|): {high | low}")
print(f"교집합 (&): {high & low}")
print(f"80점이상 & 50점미만 같은 사람: {len(high & low)}명")
```
</details>

In [ ]:
print("=== Pong 12: 집합 연산 응용 ===")

# spending_scores 중복 제거
score_set = ___(spending_scores)
print(f"spending_scores = {spending_scores}")
print(f"중복 제거       = {score_set}")
print()

# 80점 이상 고객 인덱스 집합
high = {i for i in range(len(spending_scores)) if spending_scores[i] >= 80}
# 50점 미만 고객 인덱스 집합
low  = {i for i in range(len(spending_scores)) if spending_scores[i] < 50}

print(f"80점 이상 인덱스: {high}")
print(f"50점 미만 인덱스: {low}")
print(f"합집합 (|): {high ___ low}")
print(f"교집합 (&): {high ___ low}")   # 없어야 정상
print(f"80점이상 & 50점미만 같은 사람: {len(high & low)}명")

---
# Section 6. 불리언(Boolean)과 논리 연산자

## 개념 설명

**불리언**(bool)은 `True` 또는 `False` 두 값만 가진다.

```python
print(5 > 3)    # True
print(5 == 3)   # False
```

**논리 연산자:**
```python
True and False  # False  ← 둘 다 True여야 True
True or  False  # True   ← 하나라도 True면 True
not True        # False  ← 반대
```

**비교 연산자:**
| 연산자 | 의미 | 예시 |
|:---|:---|:---|
| `==` | 같다 | `score == 100` |
| `!=` | 다르다 | `score != 0` |
| `>` `<` | 크다/작다 | `age > 30` |
| `>=` `<=` | 이상/이하 | `score >= 80` |
| `in` | 포함 | `"Korea" in countries` |

### Ping 6 — 고객 조건 판정

In [ ]:
print("=== Ping 6: 불리언과 논리 연산자 ===")

age_val    = ages[0]
score_val  = spending_scores[0]
income_val = incomes[0]

# 비교 연산
print(f"나이 > 30:       {age_val > 30}")
print(f"점수 >= 80:      {score_val >= 80}")
print(f"소득 == 15:      {income_val == 15}")
print()

# and: 둘 다 만족해야
is_vip = score_val >= 80 and income_val >= 70
print(f"VIP (점수≥80 AND 소득≥70): {is_vip}")

# or: 하나라도 만족
is_target = score_val >= 80 or income_val >= 70
print(f"타겟 (점수≥80 OR 소득≥70): {is_target}")

# not
is_young = age_val < 30
print(f"젊은 고객:     {is_young}")
print(f"젊지 않은 고객: {not is_young}")
print()

# in 연산
print(f"'Male' in genders:   {'Male' in genders}")
print(f"'Child' in genders:  {'Child' in genders}")

### Pong 13 — COVID 데이터로 다양한 불리언 조건을 판정하라.

<details>
<summary>▶ 정답 보기</summary>

```python
print("=== Pong 13: 불리언 조건 판정 ===")
kr_cases  = total_cases_list[0]
kr_deaths = total_deaths_list[0]
kr_pop    = populations[0]
kr_gdp    = gdps[0]

death_rate = kr_deaths / kr_cases * 100
vax_pct    = kr_deaths / kr_pop * 100

print(f"확진자 > 1000만:        {kr_cases > 10000000}")
print(f"사망률 < 1%:            {death_rate < 1.0}")
print(f"GDP > 3만달러:          {kr_gdp > 30000}")

is_high_risk = kr_cases > 10000000 and death_rate > 1.0
print(f"고위험 (확진>1000만 AND 사망률>1%): {is_high_risk}")

is_concern = kr_cases > 5000000 or death_rate > 2.0
print(f"주의 (확진>500만 OR 사망률>2%): {is_concern}")

is_low_cases = kr_cases < 1000000
print(f"소규모 유행:  {is_low_cases}")
print(f"대규모 유행:  {not is_low_cases}")

print(f"'South Korea' in countries: {'South Korea' in countries}")
print(f"'Brazil' in countries:      {'Brazil' in countries}")
```
</details>

In [ ]:
print("=== Pong 13: 불리언 조건 판정 ===")

kr_cases  = total_cases_list[0]
kr_deaths = total_deaths_list[0]
kr_pop    = populations[0]
kr_gdp    = gdps[0]

death_rate = kr_deaths / kr_cases * 100
vax_pct    = kr_deaths / kr_pop * 100   # 가정

# 비교 연산
print(f"확진자 > 1000만:        {kr_cases ___ 10000000}")
print(f"사망률 < 1%:            {death_rate ___ 1.0}")
print(f"GDP > 3만달러:          {kr_gdp ___ 30000}")
print()

# and / or
is_high_risk = kr_cases > 10000000 ___ death_rate > 1.0
print(f"고위험 (확진>1000만 AND 사망률>1%): {is_high_risk}")

is_concern   = kr_cases > 5000000 ___ death_rate > 2.0
print(f"주의 (확진>500만 OR 사망률>2%): {is_concern}")
print()

# not
is_low_cases = kr_cases < 1000000
print(f"소규모 유행:  {is_low_cases}")
print(f"대규모 유행:  {not is_low_cases}")

# in
print(f"'South Korea' in countries: {'South Korea' ___ countries}")
print(f"'Brazil' in countries:      {'Brazil' ___ countries}")

### Pong 14 — 불리언 값을 int로 변환하고, `True + True`가 얼마인지 확인하라.

<details>
<summary>▶ 정답 보기</summary>

```python
print("=== Pong 14: 불리언 심화 ===")
print(f"int(True)  = {int(True)}")
print(f"int(False) = {int(False)}")

print(f"True + True   = {True + True}")
print(f"True + False  = {True + False}")
print(f"True * 100    = {True * 100}")

count_high_cases = (total_cases_list[0] > 10000000) +                    (total_cases_list[1] > 10000000) +                    (total_cases_list[2] > 10000000) +                    (total_cases_list[3] > 10000000) +                    (total_cases_list[4] > 10000000)
print(f"확진자 1000만 이상 국가 수: {count_high_cases}개국")
```
</details>

In [ ]:
print("=== Pong 14: 불리언 심화 ===")

# bool → int 변환
print(f"int(True)  = {int(True)}")
print(f"int(False) = {int(False)}")
print()

# True는 1, False는 0처럼 연산됨
print(f"True + True   = {True + True}")
print(f"True + False  = {True + False}")
print(f"True * 100    = {True * 100}")
print()

# 실용 예: 조건 만족 국가 수 세기
count_high_cases = (total_cases_list[0] > 10000000) +                    (total_cases_list[1] > ___) +                    (total_cases_list[2] > 10000000) +                    (total_cases_list[3] > 10000000) +                    (total_cases_list[4] > 10000000)
print(f"확진자 1000만 이상 국가 수: {count_high_cases}개국")

---
# Section 7. 종합 실습

리스트, 튜플, 딕셔너리, 집합, 불리언을 모두 활용한다.

### Ping 7 — 쇼핑몰 고객 종합 분석 보고서

In [ ]:
print("=== Ping 7: 쇼핑몰 종합 분석 ===")

# 통계
avg_age    = sum(ages) / len(ages)
avg_income = sum(incomes) / len(incomes)
avg_score  = sum(spending_scores) / len(spending_scores)
gender_set = set(genders)

# 고객 딕셔너리 (첫 번째 고객)
top_customer = {
    "id":     customer_ids[spending_scores.index(max(spending_scores))],
    "score":  max(spending_scores),
    "gender": genders[spending_scores.index(max(spending_scores))],
    "age":    ages[spending_scores.index(max(spending_scores))]
}

line = "=" * 44
print(line)
print(f"  {'쇼핑몰 고객 분석 보고서 (상위 5명)':^28}")
print(line)
print(f"  {'총 분석 고객':<12}: {len(customer_ids)}명")
print(f"  {'성별 구성':<12}: {gender_set}")
print(f"  {'평균 나이':<12}: {avg_age:.1f}세")
print(f"  {'평균 소득':<12}: {avg_income:.1f}k$")
print(f"  {'평균 소비점수':<10}: {avg_score:.1f}점")
print(line)
print(f"  최고 소비점수 고객:")
print(f"    ID {top_customer['id']}, {top_customer['gender']}, "
      f"{top_customer['age']}세, {top_customer['score']}점")
print(line)

### Pong 15 — 5개국 COVID 데이터를 딕셔너리 리스트로 만들고 종합 보고서를 출력하라.

<details>
<summary>▶ 정답 보기</summary>

```python
print("=== Pong 15: COVID 5개국 종합 보고서 ===")
data = [
    {"country": countries[i],
     "cases":   total_cases_list[i],
     "deaths":  total_deaths_list[i],
     "pop":     populations[i],
     "gdp":     gdps[i]}
    for i in range(5)
]

total_confirmed = sum(total_cases_list)
total_dead      = sum(total_deaths_list)
continent_set   = set(continents)

line = "=" * 46
print(line)
print(f"  {'COVID-19 5개국 종합 보고서':^30}")
print(line)
print(f"  분석 국가:    {countries}")
print(f"  총 확진자:    {total_confirmed:,}명")
print(f"  총 사망자:    {total_dead:,}명")
print(f"  포함 대륙:    {continent_set}")
print(line)
print(f"  {'국가':<18} {'확진자':>12} {'사망률':>8}")
print("-" * 42)
for d in data:
    dr = d['deaths'] / d['cases'] * 100 if d['cases'] > 0 else 0
    print(f"  {d['country']:<18} {d['cases']:>12,} {dr:>7.2f}%")
print(line)
```
</details>

In [ ]:
print("=== Pong 15: COVID 5개국 종합 보고서 ===")

# 5개국 데이터를 딕셔너리로
data = [
    {"country": countries[i],
     "cases":   total_cases_list[i],
     "deaths":  total_deaths_list[i],
     "pop":     populations[i],
     "gdp":     gdps[i]}
    for i in range(5)
]

# 통계 (리스트 활용)
total_confirmed = sum(___)
total_dead      = sum(___)
continent_set   = set(___)

line = "=" * 46
print(line)
print(f"  {'COVID-19 5개국 종합 보고서':^30}")
print(line)
print(f"  분석 국가:    {countries}")
print(f"  총 확진자:    {total_confirmed:,}명")
print(f"  총 사망자:    {total_dead:,}명")
print(f"  포함 대륙:    {continent_set}")
print(line)
print(f"  {'국가':<18} {'확진자':>12} {'사망률':>8}")
print("-" * 42)
for d in data:
    dr = d['deaths'] / d['cases'] * 100 if d['cases'] > 0 else 0
    print(f"  {d['country']:<18} {d['cases']:>12,} {dr:>7.2f}%")
print(line)

### Pong 16 — 사망률이 가장 높은 국가와 가장 낮은 국가를 딕셔너리로 구성하여 출력하라.

<details>
<summary>▶ 정답 보기</summary>

```python
print("=== Pong 16: 최고/최저 사망률 국가 ===")
death_rates = [
    total_deaths_list[0] / total_cases_list[0] * 100,
    total_deaths_list[1] / total_cases_list[1] * 100,
    total_deaths_list[2] / total_cases_list[2] * 100,
    total_deaths_list[3] / total_cases_list[3] * 100,
    total_deaths_list[4] / total_cases_list[4] * 100,
]
print(f"사망률: {[round(r, 3) for r in death_rates]}")

max_idx = death_rates.index(max(death_rates))
min_idx = death_rates.index(min(death_rates))

worst = {"country": countries[max_idx], "death_rate": death_rates[max_idx], "cases": total_cases_list[max_idx]}
best  = {"country": countries[min_idx], "death_rate": death_rates[min_idx], "cases": total_cases_list[min_idx]}

print(f"사망률 최고: {worst['country']} ({worst['death_rate']:.3f}%)")
print(f"사망률 최저: {best['country']}  ({best['death_rate']:.3f}%)")
```
</details>

In [ ]:
print("=== Pong 16: 최고/최저 사망률 국가 ===")

# 사망률 리스트 계산
death_rates = [
    total_deaths_list[0] / total_cases_list[0] * 100,
    total_deaths_list[1] / total_cases_list[1] * 100,
    total_deaths_list[2] / total_cases_list[2] * 100,
    total_deaths_list[3] / total_cases_list[3] * 100,
    total_deaths_list[4] / total_cases_list[4] * 100,
]
print(f"사망률: {[round(r, 3) for r in death_rates]}")
print()

# 최고/최저 인덱스
max_idx = death_rates.index(___(death_rates))
min_idx = death_rates.index(___(death_rates))

worst = {
    "country":    countries[___],
    "death_rate": death_rates[___],
    "cases":      total_cases_list[___]
}
best  = {
    "country":    countries[___],
    "death_rate": death_rates[___],
    "cases":      total_cases_list[___]
}

print(f"사망률 최고: {worst['country']} ({worst['death_rate']:.3f}%)")
print(f"사망률 최저: {best['country']}  ({best['death_rate']:.3f}%)")

---
# Section 8. 연습문제

스스로 풀어보고 정답을 확인하라.

### 연습 1 — `incomes` 리스트의 평균, 최댓값, 최솟값을 출력하라.

<details>
<summary>▶ 정답 보기</summary>

```python
print(f'평균: {sum(incomes)/len(incomes):.1f}k$')
print(f'최고: {max(incomes)}k$')
print(f'최저: {min(incomes)}k$')
```
</details>

In [ ]:
# 여기에 직접 풀어보세요

### 연습 2 — `spending_scores`를 내림차순으로 정렬한 새 리스트를 만들어라. (원본 유지)

<details>
<summary>▶ 정답 보기</summary>

```python
desc = sorted(spending_scores, reverse=True)
print(desc)
print(spending_scores)  # 원본 유지 확인
```
</details>

In [ ]:
# 여기에 직접 풀어보세요

### 연습 3 — `countries` 리스트에 `'Brazil'`을 추가하고, `'Japan'`을 제거하라.

<details>
<summary>▶ 정답 보기</summary>

```python
c = countries.copy()
c.append('Brazil')
c.remove('Japan')
print(c)
```
</details>

In [ ]:
# 여기에 직접 풀어보세요

### 연습 4 — `genders` 리스트에서 중복을 제거하여 고유 성별 집합을 만들어라.

<details>
<summary>▶ 정답 보기</summary>

```python
unique = set(genders)
print(unique)
print(f'고유 성별 수: {len(unique)}개')
```
</details>

In [ ]:
# 여기에 직접 풀어보세요

### 연습 5 — 첫 번째 고객 정보를 딕셔너리로 만들고 `'grade'` 키를 추가하라. (score >= 70이면 'Gold', 아니면 'Silver')

<details>
<summary>▶ 정답 보기</summary>

```python
c = {'id': customer_ids[0], 'age': ages[0], 'score': spending_scores[0]}
c['grade'] = 'Gold' if c['score'] >= 70 else 'Silver'
print(c)
```
</details>

In [ ]:
# 여기에 직접 풀어보세요

### 연습 6 — `total_cases_list`에서 1000만 이상인 국가 인덱스를 집합으로 만들어라.

<details>
<summary>▶ 정답 보기</summary>

```python
big = {i for i in range(len(total_cases_list)) if total_cases_list[i] >= 10000000}
print(big)
```
</details>

In [ ]:
# 여기에 직접 풀어보세요

### 연습 7 — `(countries[0], total_cases_list[0], total_deaths_list[0])` 튜플을 만들고 언패킹하라.

<details>
<summary>▶ 정답 보기</summary>

```python
t = (countries[0], total_cases_list[0], total_deaths_list[0])
name, cases, deaths = t
print(f'{name}: 확진 {cases:,}, 사망 {deaths:,}')
```
</details>

In [ ]:
# 여기에 직접 풀어보세요

### 연습 8 — `spending_scores[0] >= 80 and ages[0] < 40`의 결과를 출력하고 의미를 주석으로 적어라.

<details>
<summary>▶ 정답 보기</summary>

```python
result = spending_scores[0] >= 80 and ages[0] < 40
print(result)  # 소비점수 높고 젊은 고객
```
</details>

In [ ]:
# 여기에 직접 풀어보세요

### 연습 9 — `countries` 딕셔너리를 만들어라. `{국가명: 확진자수}` 형태로 5개국을 모두 넣어라.

<details>
<summary>▶ 정답 보기</summary>

```python
d = {countries[i]: total_cases_list[i] for i in range(5)}
print(d)
```
</details>

In [ ]:
# 여기에 직접 풀어보세요

### 연습 10 — `total_cases_list`를 복사한 후 `append(999999)`를 수행하고 원본이 바뀌지 않았음을 확인하라.

<details>
<summary>▶ 정답 보기</summary>

```python
copy = total_cases_list.copy()
copy.append(999999)
print(f'원본: {total_cases_list}')
print(f'복사: {copy}')
```
</details>

In [ ]:
# 여기에 직접 풀어보세요